# RAG Evaluation — Live Demo
**AI Engineering · Module 1 · Lecture 3 · Part 4**

We evaluate a RAG pipeline across all four core metrics:

| Metric | What it measures | Failure it catches |
|--------|-----------------|--------------------|
| **Context Recall** | Did we retrieve all relevant chunks? | Retrieval miss |
| **Context Precision** | Are retrieved chunks actually relevant? | Retrieval noise |
| **Answer Faithfulness** | Is the answer supported by the context? | LLM hallucination |
| **Answer Relevance** | Does the answer address the question? | LLM off-topic |

> **Uses Nebius API for both embeddings and LLM-as-judge**

## 0. Setup

In [1]:
# ingnore errors in this cell
!pip install openai chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 700.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
import json
import re
import textwrap
from typing import List, Dict, Tuple

from openai import OpenAI
import chromadb

NEBIUS_API_KEY = 'YOUR_NEBIUS_API_KEY'   # ← paste your token
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"
EMBED_MODEL     = "BAAI/bge-en-icl"
JUDGE_MODEL     = "meta-llama/Meta-Llama-3.1-8B-Instruct"
RAG_MODEL       = "meta-llama/Meta-Llama-3.1-8B-Instruct"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


## 1. Build the RAG Pipeline (same as demo notebook)

In [3]:
# ── Knowledge base ────────────────────────────────────────────────────────
# We use a self-contained text so the demo works without a PDF download

KNOWLEDGE_BASE = """
Global Energy Review 2025 — Key Highlights

CO2 Emissions
Global energy-related CO2 emissions rose to a record 37.4 Gt in 2024, an increase of 0.8% from 2023.
Emerging market and developing economies drove emissions growth, while advanced economies saw a
collective decline of 3.4%, led by the European Union, United States, and Japan.
The power sector accounted for 42% of total energy-related emissions.

Renewables
Renewable energy capacity additions reached a new record of 585 GW in 2024.
Solar PV alone accounted for 420 GW of new capacity, driven by massive expansion in China.
Wind power additions totalled 130 GW, with offshore wind growing to 40 GW annually.
Renewables now supply 35% of global electricity generation.

Fossil Fuels
Coal demand reached a new record of 8.77 billion tonnes in 2024, primarily in Asia.
Natural gas demand grew by 2.5%, supported by LNG trade expansion.
Oil demand growth slowed to 0.8 mb/d, compared to 2.1 mb/d in 2023.
Electric vehicles displaced approximately 1.5 mb/d of oil demand in 2024.

Clean Energy Investment
Clean energy investment reached USD 2 trillion in 2024 for the first time.
Fossil fuel investment was USD 1 trillion, meaning clean energy received twice as much funding.
Battery storage investment tripled compared to 2022, reaching USD 150 billion.
Heat pump sales exceeded 15 million units globally.

Electricity
Global electricity demand grew by 4% in 2024, faster than total energy demand.
Data centres and AI contributed significantly to electricity demand growth in advanced economies.
Nuclear generation reached a record high of 2,800 TWh in 2024.
The share of low-carbon electricity exceeded 40% for the first time.
"""

# ── Chunking ──────────────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 350, overlap: int = 70) -> List[str]:
    text = " ".join(text.split())
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= chunk_size:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            if chunks and overlap > 0:
                prev_words = chunks[-1].split()
                current = " ".join(prev_words[-overlap//5:]) + " " + sentence
            else:
                current = sentence
    if current:
        chunks.append(current)
    return [c for c in chunks if len(c.strip()) > 30]


# ── Embedding ─────────────────────────────────────────────────────────────
def embed_texts(texts: List[str]) -> List[List[float]]:
    all_embs = []
    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embs.extend([item.embedding for item in resp.data])
    return all_embs


# ── Build index ───────────────────────────────────────────────────────────
chunks = chunk_text(KNOWLEDGE_BASE)
print(f"Chunks: {len(chunks)}")

print("Embedding...")
embeddings = embed_texts(chunks)

chroma = chromadb.Client()
try:
    chroma.delete_collection("eval_demo")
except:
    pass
collection = chroma.create_collection("eval_demo", metadata={"hnsw:space": "cosine"})
collection.add(documents=chunks, embeddings=embeddings, ids=[f"c{i}" for i in range(len(chunks))])

print(f"Index ready: {collection.count()} chunks ✓")

Chunks: 7
Embedding...
Index ready: 7 chunks ✓


In [4]:
# ── RAG pipeline functions ────────────────────────────────────────────────

def retrieve(query: str, top_k: int = 3) -> List[Dict]:
    q_emb = embed_texts([query])[0]
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "distances"]
    )
    return [
        {
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "similarity": round(1 - results["distances"][0][i], 4)
        }
        for i in range(len(results["documents"][0]))
    ]


def generate_answer(query: str, retrieved_chunks: List[Dict]) -> str:
    context = "\n\n".join(
        f"[Source {i+1}]\n{c['text']}" for i, c in enumerate(retrieved_chunks)
    )
    prompt = f"""Answer using ONLY the context below. If the answer isn't there, say so.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""
    response = client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": "Answer based strictly on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()


def rag(query: str, top_k: int = 3) -> Tuple[str, List[Dict]]:
    retrieved = retrieve(query, top_k)
    answer = generate_answer(query, retrieved)
    return answer, retrieved


# Quick test
ans, ctx = rag("How much did renewable capacity grow in 2024?")
print("Answer:", ans)

Answer: 585 GW (Source 2)


## 2. Define the Evaluation Dataset

A good eval dataset contains:
- **Question** — what we ask the RAG system
- **Ground truth answer** — the correct answer (ground truth)
- **Ground truth  chunks** — which specific chunks in the index should be retrieved

In [5]:
# Ground truth dataset: each item has a question, the correct answer,
# and the IDs of chunks that SHOULD be retrieved

# First, let's see what chunk IDs we have and what they contain
print("Available chunks:")
for i, chunk in enumerate(chunks):
    print(f"  c{i}: {chunk[:100]}...")

Available chunks:
  c0: Global Energy Review 2025 — Key Highlights CO2 Emissions Global energy-related CO2 emissions rose to...
  c1: a collective decline of 3.4%, led by the European Union, United States, and Japan. The power sector ...
  c2: alone accounted for 420 GW of new capacity, driven by massive expansion in China. Wind power additio...
  c3: demand reached a new record of 8.77 billion tonnes in 2024, primarily in Asia. Natural gas demand gr...
  c4: mb/d in 2023. Electric vehicles displaced approximately 1.5 mb/d of oil demand in 2024. Clean Energy...
  c5: fuel investment was USD 1 trillion, meaning clean energy received twice as much funding. Battery sto...
  c6: Electricity Global electricity demand grew by 4% in 2024, faster than total energy demand. Data cent...


In [6]:
# Now define the Ground truth dataset
# Adjust GT_chunk_ids after inspecting the chunk printout above

EVAL_DATASET = [
    {
        "question": "What was the total renewable energy capacity added in 2024?",
        "gt_answer": "585 GW of renewable energy capacity was added in 2024, a new record.",
        "gt_chunk_ids": ["c1", "c2"],   # chunks mentioning renewables
    },
    {
        "question": "How did clean energy investment compare to fossil fuel investment in 2024?",
        "gt_answer": "Clean energy investment reached USD 2 trillion, twice the USD 1 trillion invested in fossil fuels.",
        "gt_chunk_ids": ["c4", "c5"],
    },
    {
        "question": "What share of global electricity generation came from renewables?",
        "gt_answer": "Renewables supplied 35% of global electricity generation.",
        "gt_chunk_ids": ["c2", "c3"],
    },
    {
        "question": "How much did electric vehicles reduce oil demand?",
        "gt_answer": "Electric vehicles displaced approximately 1.5 mb/d of oil demand in 2024.",
        "gt_chunk_ids": ["c3", "c4"],
    },
    {
        "question": "What was the state of nuclear generation in 2024?",
        "gt_answer": "Nuclear generation reached a record high of 2,800 TWh in 2024.",
        "gt_chunk_ids": ["c6", "c7"],
    },
    {
        # Out-of-scope question — the RAG system should say it doesn't know
        "question": "What is the current oil price per barrel?",
        "gt_answer": "The context does not contain information about current oil prices.",
        "gt_chunk_ids": [],   # no relevant chunks exist
    },
]

print(f"Eval dataset: {len(EVAL_DATASET)} questions")

Eval dataset: 6 questions


## 3. Metric 1 & 2 — Retrieval Metrics

**Context Recall** = how many of the gt chunks did we actually retrieve?  
**Context Precision** = of everything we retrieved, how much was actually relevant?

In [7]:
def context_recall(retrieved_ids: List[str], gt_ids: List[str]) -> float:
    """
    Recall = |retrieved ∩ gt| / |gt|
    How many of the chunks we needed did we actually get?
    """
    if not gt_ids:
        return 1.0  # no gt chunks needed → trivially recall=1
    retrieved_set = set(retrieved_ids)
    gt_set = set(gt_ids)
    return len(retrieved_set & gt_set) / len(gt_set)


def context_precision(retrieved_ids: List[str], gt_ids: List[str]) -> float:
    """
    Precision = |retrieved ∩ gt| / |retrieved|
    Of what we retrieved, how much was actually useful?
    """
    if not retrieved_ids:
        return 0.0
    retrieved_set = set(retrieved_ids)
    gt_set = set(gt_ids)
    return len(retrieved_set & gt_set) / len(retrieved_set)


# Evaluate retrieval for all questions
print(f"{'Question':<50} {'Recall':>8} {'Precision':>10}")
print("-" * 72)

recall_scores, precision_scores = [], []
retrieval_results = []

for item in EVAL_DATASET:
    retrieved = retrieve(item["question"], top_k=3)
    # print(retrieved)
    retrieved_ids = [r["id"] for r in retrieved]

    recall = context_recall(retrieved_ids, item["gt_chunk_ids"])
    precision = context_precision(retrieved_ids, item["gt_chunk_ids"])

    recall_scores.append(recall)
    precision_scores.append(precision)
    retrieval_results.append({"retrieved": retrieved, "recall": recall, "precision": precision})

    q_short = item["question"][:48] + ".." if len(item["question"]) > 48 else item["question"]
    print(f"{q_short:<50} {recall:>8.2f} {precision:>10.2f}")

print("-" * 72)
print(f"{'AVERAGE':<50} {sum(recall_scores)/len(recall_scores):>8.2f} {sum(precision_scores)/len(precision_scores):>10.2f}")

Question                                             Recall  Precision
------------------------------------------------------------------------
What was the total renewable energy capacity add..     1.00       0.67
How did clean energy investment compare to fossi..     1.00       0.67
What share of global electricity generation came..     0.50       0.33
How much did electric vehicles reduce oil demand..     1.00       0.67
What was the state of nuclear generation in 2024..     0.50       0.33
What is the current oil price per barrel?              1.00       0.00
------------------------------------------------------------------------
AVERAGE                                                0.83       0.44


## 4. Metrics 3 & 4 — Generation Metrics (LLM-as-Judge)

We use the judge LLM to score:
- **Faithfulness**: Is the answer supported by the context (no hallucination)?
- **Answer Relevance**: Does the answer actually address the question?

In [8]:
FAITHFULNESS_PROMPT = """You are an evaluation assistant.
Your task is to assess whether the ANSWER is faithful to the CONTEXT —
meaning every claim in the answer is supported by the context.

CONTEXT:
{context}

QUESTION: {question}
ANSWER: {answer}

Score faithfulness from 0.0 to 1.0:
- 1.0 = every claim in the answer is directly supported by the context
- 0.5 = some claims supported, some not
- 0.0 = answer contains claims not in the context (hallucination)

Respond ONLY with valid JSON, no explanation:
{{"score": <score>, "reason": <reason>}}"""


RELEVANCE_PROMPT = """You are an evaluation assistant.
Your task is to assess whether the ANSWER actually addresses the QUESTION.

QUESTION: {question}
ANSWER: {answer}

Score answer relevance from 0.0 to 1.0:
- 1.0 = answer directly and completely addresses the question
- 0.5 = answer partially addresses the question
- 0.0 = answer is off-topic or doesn't address the question

Respond ONLY with valid JSON, no explanation:
{{"score": <score>, "reason": <reason>}}"""


def judge(prompt_template: str, **kwargs) -> Dict:
    """Call the judge LLM and parse JSON response."""
    prompt = prompt_template.format(**kwargs)
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )
    raw = response.choices[0].message.content.strip()
    # Extract JSON even if surrounded by markdown fences
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        return json.loads(match.group())


print("Running LLM-as-judge evaluations...\n")

faithfulness_scores, relevance_scores = [], []
full_results = []

for i, item in enumerate(EVAL_DATASET):
    retrieved = retrieval_results[i]["retrieved"]
    answer, _ = rag(item["question"])

    context_text = "\n\n".join(f"[Source {j+1}] {r['text']}" for j, r in enumerate(retrieved))

    faith = judge(FAITHFULNESS_PROMPT,
                  context=context_text,
                  question=item["question"],
                  answer=answer)

    relev = judge(RELEVANCE_PROMPT,
                  question=item["question"],
                  answer=answer)

    faithfulness_scores.append(faith["score"])
    relevance_scores.append(relev["score"])

    full_results.append({
        "question":    item["question"],
        "answer":      answer,
        "recall":      retrieval_results[i]["recall"],
        "precision":   retrieval_results[i]["precision"],
        "faithfulness": faith["score"],
        "faith_reason": faith.get("reason", ""),
        "relevance":   relev["score"],
        "relev_reason": relev.get("reason", ""),
    })

    q_short = item["question"][:40] + ".."
    print(f"[{i+1}/{len(EVAL_DATASET)}] {q_short}")
    print(f"  Faithfulness: {faith['score']:.2f}  | {faith.get('reason','')}")
    print(f"  Relevance:    {relev['score']:.2f}  | {relev.get('reason','')}")
    print()

Running LLM-as-judge evaluations...

[1/6] What was the total renewable energy capa..
  Faithfulness: 1.00  | All claims in the answer are directly supported by the context in Source 1.
  Relevance:    1.00  | The answer directly states a specific number, which is a complete answer to the question about the total renewable energy capacity added in 2024.

[2/6] How did clean energy investment compare ..
  Faithfulness: 1.00  | All claims in the answer are directly supported by the context in Source 1.
  Relevance:    1.00  | The answer directly compares clean energy investment to fossil fuel investment in 2024, providing a specific numerical value for both.

[3/6] What share of global electricity generat..
  Faithfulness: 1.00  | The answer directly matches the claim in Source 1 that renewables now supply 35% of global electricity generation.
  Relevance:    1.00  | answer directly and completely addresses the question

[4/6] How much did electric vehicles reduce oi..
  Faithfulness: 1.

## 5. Full Evaluation Summary

In [9]:
print("=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
print()

def avg(lst): return sum(lst) / len(lst) if lst else 0

metrics = [
    ("Context Recall",     recall_scores,       "Retrieval"),
    ("Context Precision",  precision_scores,    "Retrieval"),
    ("Answer Faithfulness",faithfulness_scores, "Generation"),
    ("Answer Relevance",   relevance_scores,    "Generation"),
]

print(f"{'Metric':<25} {'Category':<12} {'Average':>8} {'Min':>6} {'Max':>6}")
print("-" * 65)
for name, scores, category in metrics:
    print(f"{name:<25} {category:<12} {avg(scores):>8.3f} {min(scores):>6.2f} {max(scores):>6.2f}")

overall = avg([
    avg(recall_scores),
    avg(precision_scores),
    avg(faithfulness_scores),
    avg(relevance_scores)
])
print("-" * 65)
print(f"{'OVERALL SCORE':<25} {'':12} {overall:>8.3f}")

print()
print("Per-question breakdown:")
print(f"{'#':<3} {'Question':<38} {'Rec':>5} {'Pre':>5} {'Fai':>5} {'Rel':>5}")
print("-" * 65)
for i, r in enumerate(full_results):
    q_short = r["question"][:36] + ".." if len(r["question"]) > 36 else r["question"]
    print(f"{i+1:<3} {q_short:<38} {r['recall']:>5.2f} {r['precision']:>5.2f} {r['faithfulness']:>5.2f} {r['relevance']:>5.2f}")

EVALUATION SUMMARY

Metric                    Category      Average    Min    Max
-----------------------------------------------------------------
Context Recall            Retrieval       0.833   0.50   1.00
Context Precision         Retrieval       0.444   0.00   0.67
Answer Faithfulness       Generation      1.000   1.00   1.00
Answer Relevance          Generation      0.833   0.00   1.00
-----------------------------------------------------------------
OVERALL SCORE                             0.778

Per-question breakdown:
#   Question                                 Rec   Pre   Fai   Rel
-----------------------------------------------------------------
1   What was the total renewable energy ..  1.00  0.67  1.00  1.00
2   How did clean energy investment comp..  1.00  0.67  1.00  1.00
3   What share of global electricity gen..  0.50  0.33  1.00  1.00
4   How much did electric vehicles reduc..  1.00  0.67  1.00  1.00
5   What was the state of nuclear genera..  0.50  0.33  1.00  1.

## 6. Diagnose — Where is the pipeline failing?

In [10]:
print("DIAGNOSTIC: Cases that need attention\n")
print("="*80)

THRESHOLD = 0.7

for r in full_results:
    issues = []
    if r["recall"] < THRESHOLD:      issues.append(f"LOW RECALL ({r['recall']:.2f}) — gt chunks not retrieved")
    if r["precision"] < THRESHOLD:   issues.append(f"LOW PRECISION ({r['precision']:.2f}) — too much noise retrieved")
    if r["faithfulness"] < THRESHOLD: issues.append(f"LOW FAITHFULNESS ({r['faithfulness']:.2f}) — {r['faith_reason']}")
    if r["relevance"] < THRESHOLD:   issues.append(f"LOW RELEVANCE ({r['relevance']:.2f}) — {r['relev_reason']}")

    if issues:
        print(f"Q: {r['question']}")
        print(f"A: {r['answer'][:200]}..." if len(r['answer']) > 200 else f"A: {r['answer']}")
        for issue in issues:
            print(f"  ⚠️  {issue}")
        print()

if not any(
    r["recall"] < THRESHOLD or r["precision"] < THRESHOLD or
    r["faithfulness"] < THRESHOLD or r["relevance"] < THRESHOLD
    for r in full_results
):
    print(f"All metrics above {THRESHOLD} threshold. Pipeline looks good! ✓")

DIAGNOSTIC: Cases that need attention

Q: What was the total renewable energy capacity added in 2024?
A: 585 GW
  ⚠️  LOW PRECISION (0.67) — too much noise retrieved

Q: How did clean energy investment compare to fossil fuel investment in 2024?
A: Clean energy investment was USD 2 trillion, while fossil fuel investment was USD 1 trillion, meaning clean energy received twice as much funding.
  ⚠️  LOW PRECISION (0.67) — too much noise retrieved

Q: What share of global electricity generation came from renewables?
A: 35%
  ⚠️  LOW RECALL (0.50) — gt chunks not retrieved
  ⚠️  LOW PRECISION (0.33) — too much noise retrieved

Q: How much did electric vehicles reduce oil demand?
A: 1.5 mb/d
  ⚠️  LOW PRECISION (0.67) — too much noise retrieved

Q: What was the state of nuclear generation in 2024?
A: Nuclear generation reached a record high of 2,800 TWh in 2024.
  ⚠️  LOW RECALL (0.50) — gt chunks not retrieved
  ⚠️  LOW PRECISION (0.33) — too much noise retrieved

Q: What is the current oil

## 7. Experiment: Does top-k affect the scores?

This is the key iteration loop — tuning retrieval parameters and watching the metrics respond.

In [11]:
print("Experiment: top-k vs. retrieval metrics")
print(f"{'top_k':<8} {'Recall':>8} {'Precision':>10} {'Trade-off':<30}")
print("-" * 60)

for top_k in [1, 2, 3, 5]:
    recalls, precisions = [], []
    for item in EVAL_DATASET:
        if not item["gt_chunk_ids"]:
            continue
        retrieved = retrieve(item["question"], top_k=top_k)
        r_ids = [r["id"] for r in retrieved]
        recalls.append(context_recall(r_ids, item["gt_chunk_ids"]))
        precisions.append(context_precision(r_ids, item["gt_chunk_ids"]))

    r_avg = avg(recalls)
    p_avg = avg(precisions)

    if r_avg >= 0.8 and p_avg >= 0.5:
        note = "✓ Good balance"
    elif r_avg >= 0.8:
        note = "High recall, lots of noise"
    elif p_avg >= 0.7:
        note = "Precise but missing some"
    else:
        note = "Low on both"

    print(f"{top_k:<8} {r_avg:>8.3f} {p_avg:>10.3f} {note}")

Experiment: top-k vs. retrieval metrics
top_k      Recall  Precision Trade-off                     
------------------------------------------------------------
1           0.500      1.000 Precise but missing some
2           0.800      0.800 ✓ Good balance
3           0.800      0.533 ✓ Good balance
5           0.800      0.320 High recall, lots of noise


---
## Summary: What we built

```
Query
  │
  ├── Retrieve (top-k chunks by cosine similarity)
  │     ↓
  │   Measure: Context Recall + Context Precision
  │
  ├── Generate (RAG model with grounding prompt)
  │     ↓
  │   Measure: Faithfulness + Relevance (LLM-as-Judge)
  │
  └── Diagnose: where is the pipeline failing?
```

**Key takeaways:**
- You need **separate metrics for retrieval and generation** — a good retriever can mask a bad generator and vice versa
- **LLM-as-judge** gives you scalable generation quality scores without human annotation
- **top-k is a retrieval hyperparameter** — higher k = higher recall, lower precision
- The out-of-scope question reveals whether grounding is working (model says "I don't know")

**Next steps (try yourself):**
- Swap in a different embedding model — does recall improve?
- Add query rephrasing before retrieval — does precision improve?
- Change chunk size (from 200 → 600) — which metrics are most affected?